# 08 • Datas, calendário e janelas

## Preparar esta sessão
Execute as células abaixo antes de começar a aula. Elas localizam os arquivos e preparam os objetos necessários. Você não precisa decorar essa preparação agora.

In [ ]:
from pathlib import Path
import pandas as pd
import math
BASE = Path.cwd()
if not (BASE / 'dados').exists() and (BASE.parent / 'dados').exists():
    BASE = BASE.parent
DADOS = BASE / 'dados'
SAIDAS = BASE / 'resultados'
SAIDAS.mkdir(exist_ok=True)
assert (DADOS/'penguins.csv').exists(), 'Descompacte o curso inteiro e abra na pasta correta.'


## Preparação independente
Execute esta célula para recriar os objetos necessários dos módulos anteriores.

In [ ]:
p = pd.read_csv(DADOS / 'penguins.csv')
print(p.head(3).to_string(index=False))
print('Dimensões:', p.shape)
print('Colunas:', p.columns.tolist())
print(type(p['species']).__name__)

mascara = p['species'].eq('Adelie') & p['body_mass_g'].gt(4000)
print(p.loc[mascara, ['species', 'island', 'body_mass_g']].head())
print(p.iloc[:3, :2])

qualidade = pd.DataFrame({'ausentes':p.isna().sum(), 'percentual':p.isna().mean().mul(100)})
print(qualidade)
medidos = p.dropna(subset=['body_mass_g']).copy()
print('Observações com massa:', len(medidos))

enriquecido = p.assign(body_mass_kg=p['body_mass_g']/1000)
print(enriquecido[['body_mass_g','body_mass_kg']].head())
print(p['species'].astype('string').str.lower().head())

resumo = p.groupby('species', as_index=False, observed=True).agg(
    observacoes=('body_mass_g','size'),
    massas_validas=('body_mass_g','count'),
    massa_media_g=('body_mass_g','mean'))
print(resumo)

nomes = pd.DataFrame({'species':['Adelie','Chinstrap','Gentoo'],
                      'nome_pt':['Adélia','Barbicha','Papua']})
cruzado = p.merge(nomes, on='species', how='left', validate='many_to_one', indicator=True)
print(cruzado['_merge'].value_counts())

r = pd.read_csv(DADOS/'online_retail.csv', dtype={
    'InvoiceNo':'string','StockCode':'string','CustomerID':'string'})
r['InvoiceDate'] = pd.to_datetime(r['InvoiceDate'], errors='coerce')
cancelada = r['InvoiceNo'].str.upper().str.startswith('C', na=False)
elegivel = (r['Quantity'].gt(0) & r['UnitPrice'].gt(0)
            & r['InvoiceDate'].notna() & ~cancelada)
v = r.loc[elegivel].copy()
v['valor'] = v['Quantity'] * v['UnitPrice']
print('Linhas brutas:',len(r),'Linhas elegíveis:',len(v))
print(r.isna().sum())

### Uma data é mais que um texto

O texto '2026-01-31' pode ser exibido, mas operações de calendário precisam de um tipo de data. to_datetime faz essa conversão. Depois, `v['InvoiceDate'].dt.year` extrai o ano; `.dt.to_period('M')` identifica o ano-mês. Agrupar somente por nome ou número do mês mistura janeiro de anos diferentes; use ano e mês juntos.

No exemplo guiado, set_index coloca InvoiceDate como índice temporário para resample. O código não substitui v. resample('MS') cria grupos mensais rotulados pelo início de cada mês; sum soma valores dentro de cada grupo. min_count=1 evita que um grupo totalmente sem valores presentes seja apresentado automaticamente como zero.

Para entender uma janela móvel, imagine valores diários fictícios 10, 20 e 30. Uma janela de duas observações produz primeiro ausência, depois 15 e depois 25, quando exige duas observações. Ela desliza pelo calendário. Antes de interpretar “sete linhas” como “sete dias”, confirme que existe uma linha por dia. shift(1) desloca o valor anterior para a linha atual, permitindo calcular (atual/anterior − 1) × 100. O primeiro período não tem anterior e deve ficar sem variação calculada.

### Aprofundamento e cuidados

`to_datetime` converte datas; o acessor `dt` extrai mês, dia e hora. Datas ambíguas como 03/04/2026 precisam de um formato explícito, por exemplo `%d/%m/%Y`. Na base da loja, trabalhamos com o horário registrado, sem informação de fuso; não inventamos uma conversão para UTC.

`resample` agrupa por intervalos de tempo quando o índice é de datas. Usaremos `MS`, início do mês. `rolling(7)` usa sete observações; `rolling("7D")` usa uma janela de tempo. Para comparar dias, reindexe o calendário e declare se ausência significa zero vendas ou falha na coleta. Aqui preencheremos com zero dentro do intervalo observado, como hipótese do exercício.

O primeiro e o último mês podem estar incompletos. A base termina em dezembro de 2011 antes do fim do mês; comparar seu total com novembro pode sugerir uma queda artificial. Crescimento percentual não é definido quando o valor anterior é zero. Valores monetários em float são adequados para esta exploração, mas sistemas financeiros exigem política explícita de precisão e arredondamento.


## Exemplo guiado

In [ ]:
mensal = v.set_index('InvoiceDate')['valor'].resample('MS').sum(min_count=1)
print(mensal)
print('Cobertura:',r['InvoiceDate'].min(),r['InvoiceDate'].max())

## Exercício 08.1
Some receita elegível por ano e mês usando dt.to_period.

In [ ]:
# Escreva sua resposta aqui


## Exercício 08.2
Calcule a receita por dia e complete o calendário com zero, documentando a hipótese.

In [ ]:
# Escreva sua resposta aqui


## Exercício 08.3
Calcule a média móvel de sete dias, exigindo sete dias para produzir resultado.

In [ ]:
# Escreva sua resposta aqui


## Exercício 08.4
Calcule variação mensal e substitua resultados com denominador anterior zero por ausência.

In [ ]:
# Escreva sua resposta aqui


## Exercício 08.5
Converta ["31/01/2026", "data ruim"] com formato brasileiro e conte falhas.

In [ ]:
# Escreva sua resposta aqui
